In [1]:
import sys
import os
import MetaTrader5 as mt5
import time

project_dir = os.path.abspath("..")
if project_dir not in sys.path:
    sys.path.append(project_dir)

import polars as pl

from src.infra.mtBase import mtBase

In [2]:
mtb = mtBase(
    account="darwinexzero_acc",
    credentials_path=os.path.join("..", "secrets", "mt5_acc_cred.yaml"),
    config_path=os.path.join("..", "secrets", "mt5_config.ini"),
)
mtb.mt5_init()

MetaTrader 5 connection established


In [6]:
info_msft = mtb.get_symbol_info("MSFT")
info_clf = mtb.get_symbol_info("CLF")

print(info_clf["trade_mode"])
print(info_msft["trade_mode"])

3
4


In [4]:
info_msft

{'symbol': 'MSFT',
 'visible': True,
 'trade_mode': 4,
 'digits': 2,
 'point': 0.01,
 'time': 0,
 'volume': 0,
 'volume_real': 0.0,
 'trade_contract_size': 1.0,
 'volume_min': 1.0,
 'volume_max': 1000.0,
 'volume_step': 1.0,
 'currency_base': 'USD',
 'currency_profit': 'USD',
 'currency_margin': 'USD',
 'trade_stops_level': 0,
 'trade_freeze_level': 0,
 'filling_mode': 3,
 'expiration_mode': 15,
 'category': ''}

In [7]:
mtb.get_account_info()._asdict()

{'login': 4000075958,
 'trade_mode': 2,
 'leverage': 1,
 'limit_orders': 400,
 'margin_so_mode': 0,
 'trade_allowed': True,
 'trade_expert': True,
 'margin_mode': 2,
 'currency_digits': 2,
 'fifo_close': False,
 'balance': 1003710.53,
 'credit': 0.0,
 'profit': 15810.9,
 'equity': 1019521.43,
 'margin': 783124.04,
 'margin_free': 220586.49,
 'margin_level': 130.18645551986884,
 'margin_so_call': 0.0,
 'margin_so_so': 0.0,
 'margin_initial': 0.0,
 'margin_maintenance': 0.0,
 'assets': 0.0,
 'liabilities': 0.0,
 'commission_blocked': 0.0,
 'name': 'kimeri_MT5',
 'server': 'Darwinex-Live',
 'currency': 'USD',
 'company': 'Tradeslide Trading Tech Limited'}

In [ ]:
mtb.get_symbol_price("MSFT")

In [7]:
symbols = mt5.symbols_get()
symbols_dict = [s._asdict() for s in symbols]
symbols_name = [s.name for s in symbols if s.trade_mode == mt5.SYMBOL_TRADE_MODE_FULL]

In [8]:
symbols_stocks = [s for s in symbols_dict if "Stock" in s['path'] and s['trade_mode'] == mt5.SYMBOL_TRADE_MODE_FULL]

print(f"Total symbols: {len(symbols)}")
print(f"Total stock symbols: {len(symbols_stocks)}")

stocks = [s['name'] for s in symbols_stocks]
for s in stocks:
    print(s)

Total symbols: 811
Total stock symbols: 687
ACHC
ADBE
ADI
ADP
ADSK
AKAM
ALGN
ALNY
AMAT
AMD
AMGN
AMKR
AMZN
APA
APLS
ARCC
ARWR
AVGO
AXON
AZTA
BIIB
BKNG
BL
BLDR
BLK
BMRN
BRKR
CACC
CAR
CASY
CDNS
CDW
CG
CGNX
CHDN
CHRW
CHTR
CINF
CMCSA
CME
COLM
COST
CPRT
CROX
CRWD
CSGP
CSX
CTAS
CTSH
DBX
DDOG
DLTR
DNLI
DOCU
DOX
DXCM
EA
EBAY
EEFT
ENPH
ENTG
ETSY
EWBC
EXEL
A
AA
AAP
ABBV
ABT
ACM
ACN
ADM
AEP
AES
AFG
AFL
AGCO
AIG
AIZ
AJG
AL
ALB
ALK
ALL
ALLY
AME
AMG
AMP
AMT
AN
ANET
AON
AOS
APD
APH
ARES
ARMK
ARW
ASH
AVTR
AVY
AWI
AWK
AXTA
AYI
AZO
BAC
BAH
BALL
BAX
BBY
BC
BDX
BEN
BFAM
BILL
BIO
BJ
BK
BKR
BLD
BMY
BR
BRKb
BRO
BSX
BURL
BWA
BX
BYD
C
CABO
CACI
CAG
CAH
CARR
CB
CBRE
CC
CCI
CCK
CCL
CE
CF
CFG
CFR
CHD
CHE
CHWY
CI
CIEN
CL
CLX
CMA
CMG
CMI
CMS
CNC
CNP
COF
COHR
COO
COP
COR
CPAY
CRL
CRM
CSL
CTLT
CTVA
CVNA
CVS
D
DAL
DAR
DAY
DD
DE
DECK
DELL
DG
DGX
DHI
DHR
DKS
DLB
DOV
DPZ
DRI
DT
DTE
DUK
DVA
DVN
DXC
ECL
EFX
EHC
EIX
EL
ELV
EME
EMN
EMR
ENOV
EOG
EPAM
EQH
EQT
ES
ESI
ESNT
ESTC
ETN
ETR
EVR
EVRG
EW
EXC
EXP
EXPD
EXPE
FANG
FAST
FFIV

In [10]:
spreads = {}
for stock in stocks:
    n_retries = 5
    for i in range(n_retries):
        mt5.symbol_select(stock, True)
        time.sleep(0.2)
        tick = mt5.symbol_info_tick(stock)._asdict()
        mt5.symbol_select(stock, False)
        time.sleep(0.1)  # to avoid overloading MT5 API
        if tick is None:
            _, le = mt5.last_error()
            print(f"Tick data not available for symbol: {stock}, error: {le}")
            continue

        # check that tick has 'ask' and 'bid' keys
        if 'ask' not in tick or 'bid' not in tick:
            print(f"Tick data incomplete for symbol: {stock}")
            continue

        ask = tick.get('ask', None)
        bid = tick.get('bid', None)
        if ask is None or bid is None:
            print(f"Tick data incomplete for symbol: {stock}")
            continue

        rel_spread = (ask - bid) / (bid + 1e-8)
        if rel_spread <= 1e-6:
            _, le = mt5.last_error()
            print(f"Unrealistic spread for symbol: {stock}, ask: {ask}, bid: {bid}, spread: {rel_spread}, error: {le}")
            continue
        else:
            break
    print(f"Symbol: {stock}, Ask: {ask}, Bid: {bid}, Spread: {rel_spread}")

    spreads[stock] = rel_spread


Symbol: ACHC, Ask: 21.36, Bid: 21.1, Spread: 0.012322274875676551
Symbol: ADBE, Ask: 334.89, Bid: 334.54, Spread: 0.0010462127099584615
Symbol: ADI, Ask: 233.14, Bid: 231.03, Spread: 0.009133013028215621
Symbol: ADP, Ask: 259.2, Bid: 258.82, Spread: 0.0014682018390592437
Symbol: ADSK, Ask: 301.47, Bid: 301.08, Spread: 0.0012953367875218873
Symbol: AKAM, Ask: 74.1, Bid: 73.5, Spread: 0.008163265305011724
Symbol: ALGN, Ask: 138.4, Bid: 136.67, Spread: 0.01265822784717521
Symbol: ALNY, Ask: 430.08, Bid: 428.0, Spread: 0.004859813083998565
Symbol: AMAT, Ask: 233.88, Bid: 233.3, Spread: 0.0024860694383845834
Symbol: AMD, Ask: 252.29, Bid: 252.1, Spread: 0.0007536691788673583
Symbol: AMGN, Ask: 298.21, Bid: 296.55, Spread: 0.005597706963223709
Symbol: AMKR, Ask: 35.89, Bid: 35.85, Spread: 0.001115760111264757
Symbol: AMZN, Ask: 251.91, Bid: 251.78, Spread: 0.0005163237747034404
Symbol: APA, Ask: 21.98, Bid: 21.95, Spread: 0.001366742596188324
Symbol: APLS, Ask: 20.42, Bid: 20.16, Spread: 0.0

In [11]:
# save dict as csv
df_spreads = pl.DataFrame({
    "symbol": list(spreads.keys()),
    "rel_spread": list(spreads.values())
})
df_spreads.write_csv("stock_spreads.csv")

In [12]:
spreads_arr = df_spreads['rel_spread'].to_numpy()
print(f"Average stock spread: {spreads_arr.mean()}")
for q in [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
    print(f"{q} quantile stock spread: {pl.Series(spreads_arr).quantile(q)}")

Average stock spread: 0.0072246561681914975
0.05 quantile stock spread: 0.0007015785517061917
0.1 quantile stock spread: 0.0010010010006670379
0.25 quantile stock spread: 0.002008298022576639
0.5 quantile stock spread: 0.0050435971958862295
0.75 quantile stock spread: 0.010437575268880518
0.9 quantile stock spread: 0.01646839488563265
0.95 quantile stock spread: 0.02131066085014579


In [13]:
mask = spreads_arr < 0.0015
filtered_df = df_spreads.filter(mask)
filtered_df.write_csv("stock_spreads_below_median.csv")

In [14]:
filtered_sym = filtered_df['symbol']
for s in filtered_sym:
    print(s)

ADBE
ADP
ADSK
AMD
AMKR
AMZN
APA
ARCC
AVGO
BLK
CMCSA
COST
CSX
CTSH
EA
EBAY
AL
ALL
BAC
BEN
BWA
C
CAH
CARR
CB
CCL
CMS
CNC
CRM
CTLT
DAL
DAY
DD
DELL
DVN
F
FCX
FE
FITB
FOXA
GEN
GIS
GM
GOOG
GOOGL
HBAN
HAL
HOLX
HPQ
HUN
ICE
IP
IPG
KHC
LKQ
LRCX
K
KDP
KEY
KMB
LLY
LUV
MA
MDLZ
MDT
MGM
META
MO
MRNA
MS
MSTR
MU
NCLH
NDAQ
NEE
NI
NFLX
NVDA
NWSA
ORCL
OXY
PGR
PINS
PPL
PYPL
QCOM
RF
REGN
RTX
SBUX
SLM
SLB
SYF
T
TFC
TSCO
TSLA
TTD
TXN
AAPL
BA
CSCO
CVX
DIS
DOW
GS
INTC
JNJ
KO
MCD
MRK
MSFT
NKE
PFE
UNH
V
VZ
WMT
XOM
TSN
UNP
UPS
USB
WEC
WFC
WHR
WM
WTRG
XEL
COIN
